# 04 — Test vol-engine (WebSocket bridge end-to-end)

Smoke test du container `fxvol-vol-engine` — étape 4/5. Valide la **chaîne complète** depuis l'engine jusqu'à un client WebSocket type frontend :

```
engine vol-engine (cycle 180s)
    │ publish_vol_update → channel vol_update (Redis pub/sub)
    ▼
api/ws/redis_bridge.py — SUBSCRIBE vol_update → broadcast WS
    │ forward via /ws/vol (cf. src/api/routers/ws.py)
    ▼
nginx — proxy_pass WebSocket sur /ws/ → api:8000
    ▼
Ce notebook (websocket-client) — connect ws://localhost/ws/vol
```

Si ce test passe → le hook frontend type `useVolStream` (smile chart, signals table, term structure) est validé en amont.

**Différence avec risk/04** : cycle vol = 180s, donc on attend **1 message en 200s** au lieu de 2-3 en 5s. Sinon structure identique.

**Couvre** :
1. Container `fxvol-api` running + healthy (gate)
2. Container `fxvol-nginx` running + healthy (gate)
3. Connect `ws://localhost/ws/vol` réussit
4. Réception ≥ 1 message JSON valide en ≤ 200s (early-exit)
5. Schéma identique au pub/sub Redis : `{symbol, timestamp, surface}`
6. Surface payload : ≥ 1 tenor public + `_svi` non-vide

**Préreq** :
- Notebooks 01-03 verts
- Stack complète : `docker compose --profile engines --profile ib up -d`
- `pip install websocket-client`

## Setup

Pas de seed — vol-engine a besoin de `latest_spot:EURUSD` (publié par market-data) et de la chain IB. Si ces deux préreq sont OK (notebooks 01-03 verts), on n'a rien à faire ici à part s'assurer qu'un spot est en place pour la durée du test.

In [1]:
import json
import subprocess
import time
from datetime import datetime

import redis

try:
    import websocket
except ImportError:
    raise RuntimeError("package `websocket-client` non installé.\n  -> python -m pip install websocket-client")

WS_URL = "ws://localhost/ws/vol"
REDIS_URL = "redis://localhost:6380/0"
SYMBOL = "EURUSD"

MIN_MESSAGES = 1
MAX_LISTEN_S = 200.0
RECV_TIMEOUT_S = 1.0

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

def docker_inspect(container, fmt):
    out = subprocess.run(["docker", "inspect", "-f", fmt, container],
                         capture_output=True, text=True)
    return out.stdout.strip()

r = redis.from_url(REDIS_URL, decode_responses=True)
spot_raw = r.get(f"latest_spot:{SYMBOL}")
if spot_raw is None:
    print("  [WARN] latest_spot absent — seed factice 1.17000")
    r.set(f"latest_spot:{SYMBOL}", "1.17000", ex=600)
else:
    print(f"  [INFO] latest_spot = {spot_raw[:40]}")
print(f"target = {WS_URL}")

  [INFO] latest_spot = 1.170195
target = ws://localhost/ws/vol


## 1. Gate — `fxvol-api` running + healthy

Le bridge Redis→WS vit dans le container api. Si api est down, `/ws/vol` ne répond pas.

In [2]:
print("== 1. fxvol-api running + healthy ==")

state = docker_inspect("fxvol-api", "{{.State.Status}}")
record("fxvol-api state running", state == "running", state or "<not found>")

health = docker_inspect("fxvol-api", "{{.State.Health.Status}}")
record("fxvol-api healthcheck", health == "healthy", health or "<no healthcheck>")

== 1. fxvol-api running + healthy ==
  [OK  ] fxvol-api state running  | running
  [OK  ] fxvol-api healthcheck  | healthy


## 2. Gate — `fxvol-nginx` running + healthy

Nginx proxy `/ws/` vers api:8000. Si nginx down, on ne peut pas joindre l'api depuis l'host.

In [3]:
print("== 2. fxvol-nginx running + healthy ==")

state = docker_inspect("fxvol-nginx", "{{.State.Status}}")
record("fxvol-nginx state running", state == "running", state or "<not found>")

health = docker_inspect("fxvol-nginx", "{{.State.Health.Status}}")
record("fxvol-nginx healthcheck", health == "healthy", health or "<no healthcheck>")

== 2. fxvol-nginx running + healthy ==
  [OK  ] fxvol-nginx state running  | running
  [OK  ] fxvol-nginx healthcheck  | healthy


## 3. Connect WebSocket `/ws/vol`

In [4]:
print("== 3. WebSocket connect ==")

ws = None
try:
    ws = websocket.create_connection(WS_URL, timeout=RECV_TIMEOUT_S)
    record("connect ws://localhost/ws/vol", ws.connected,
           f"connected = {ws.connected}")
except Exception as e:
    record("connect ws://localhost/ws/vol", False, f"{type(e).__name__}: {e}")

== 3. WebSocket connect ==
  [OK  ] connect ws://localhost/ws/vol  | connected = True


## 4. Réception ≥ 1 message en ≤ 200s (early-exit)

Cycle vol = 180s. Selon le timing du subscribe vs le prochain cycle, le message arrive en 0-180s. Early-exit dès le 1er message reçu pour ne pas faire perdre du temps.

In [5]:
print(f"== 4. réception (max {MAX_LISTEN_S}s, early-exit au 1er msg) ==")

raw_messages = []
if ws and ws.connected:
    deadline = time.perf_counter() + MAX_LISTEN_S
    start = time.perf_counter()
    while time.perf_counter() < deadline and len(raw_messages) < MIN_MESSAGES:
        try:
            data = ws.recv()
            if data:
                raw_messages.append(data)
        except websocket.WebSocketTimeoutException:
            continue
        except Exception as e:
            print(f"  [WARN] WS recv error: {type(e).__name__}: {e}")
            break
    elapsed = time.perf_counter() - start
else:
    elapsed = 0.0

record(f"≥ {MIN_MESSAGES} message reçu en ≤ {MAX_LISTEN_S}s",
       len(raw_messages) >= MIN_MESSAGES,
       f"{len(raw_messages)} message(s) en {elapsed:.1f}s")

== 4. réception (max 200.0s, early-exit au 1er msg) ==
  [OK  ] ≥ 1 message reçu en ≤ 200.0s  | 1 message(s) en 65.8s


## 5. Schéma + surface — identique au pub/sub Redis

Le bridge ne transforme pas le payload. Format attendu : `{symbol, timestamp, surface:{...}}`. La surface contient au moins 1 tenor public + `_svi`. Le détail SVI/SSVI/_fair_q est validé en notebook 02 ; ici on s'assure que **le bridge transmet la surface complète** (pas une version dégradée).

In [6]:
print("== 5. schéma + surface ==")

if not raw_messages:
    record("schéma + surface", False, "skip (cf. §4)")
else:
    parse_errors = 0
    schema_errors = 0
    parsed = []
    for raw in raw_messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            parse_errors += 1
            continue
        if {"symbol", "timestamp", "surface"} <= set(obj.keys()) and isinstance(obj.get("surface"), dict):
            parsed.append(obj)
        else:
            schema_errors += 1
    record("tous JSON-parseables",
           parse_errors == 0,
           f"{parse_errors} erreurs / {len(raw_messages)}")
    record("top-level keys = {symbol, timestamp, surface}",
           schema_errors == 0,
           f"{schema_errors} schémas incomplets / {len(raw_messages)}")

    if parsed:
        sample = parsed[0]
        record("sample.symbol == 'EURUSD'",
               sample.get("symbol") == "EURUSD",
               f"got {sample.get('symbol')!r}")
        # timestamp ISO
        try:
            datetime.fromisoformat(sample["timestamp"].replace("Z", "+00:00"))
            ts_ok = True
        except (ValueError, AttributeError, KeyError):
            ts_ok = False
        record("sample.timestamp ISO-8601 parsable", ts_ok,
               f"ts = {sample.get('timestamp')!r}")
        # surface contenu
        surface = sample.get("surface", {})
        public_tenors = [t for t in surface if not t.startswith("_") and isinstance(surface[t], dict)]
        record("≥ 1 tenor public dans surface du payload",
               len(public_tenors) >= 1,
               f"tenors = {public_tenors}")
        svi = surface.get("_svi") or {}
        record("_svi non-vide dans payload",
               bool(svi),
               f"svi tenors = {sorted(svi)}")

== 5. schéma + surface ==
  [OK  ] tous JSON-parseables  | 0 erreurs / 1
  [OK  ] top-level keys = {symbol, timestamp, surface}  | 0 schémas incomplets / 1
  [OK  ] sample.symbol == 'EURUSD'  | got 'EURUSD'
  [OK  ] sample.timestamp ISO-8601 parsable  | ts = '2026-04-29T09:30:40.486599Z'
  [OK  ] ≥ 1 tenor public dans surface du payload  | tenors = ['1M', '2M', '3M', '4M', '5M', '6M']
  [OK  ] _svi non-vide dans payload  | svi tenors = ['1M', '2M', '3M', '4M', '5M', '6M']


## Cleanup

In [7]:
if ws:
    try:
        ws.close()
        print("WebSocket closed cleanly.")
    except Exception as e:
        print(f"WS close warning: {type(e).__name__}: {e}")

WebSocket closed cleanly.


## Récap final

In [8]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<60} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<60} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — chaîne vol-engine → Redis → api → nginx → WS validée bout-à-bout.")
    print("Pass au notebook 05 (resilience).")


LABEL                                                        STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
fxvol-api state running                                      OK      running
fxvol-api healthcheck                                        OK      healthy
fxvol-nginx state running                                    OK      running
fxvol-nginx healthcheck                                      OK      healthy
connect ws://localhost/ws/vol                                OK      connected = True
≥ 1 message reçu en ≤ 200.0s                                 OK      1 message(s) en 65.8s
tous JSON-parseables                                         OK      0 erreurs / 1
top-level keys = {symbol, timestamp, surface}                OK      0 schémas incomplets / 1
sample.symbol == 'EURUSD'                                    OK      got 'EURUSD'
sample.timestamp ISO-8601 parsable                           OK     

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `fxvol-api unhealthy` | Postgres/Redis pas joignables, alembic plante | `docker logs fxvol-api --tail 50` |
| `fxvol-nginx unhealthy` | api upstream pas joignable | `docker compose restart nginx` |
| `connect WS ConnectionError` | nginx/api down | `docker compose ps` |
| `connect OK + 0 message en 200s` | bridge pas SUBSCRIBE à `vol_update` OU engine ne publie pas | grep `CH_VOL_UPDATE` dans `src/api/ws/redis_bridge.py:_FORWARDED` ; check notebook 03 d'abord |
| `schema_errors > 0` | bridge transforme le payload | inspecter `redis_bridge.py:_serve` |
| `_svi vide` ou tenors vides | engine cycle skip silencieusement (cf. notebook 02 §3) | check Trusted IPs Gateway, weekend, perms market data |
| Reçu 1 message puis dropper la connexion | nginx WS proxy timeout | check `infrastructure/nginx/nginx-dev.conf` directives `proxy_read_timeout` |